In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, f1_score, roc_auc_score
import xgboost as xgb
import os

# --- Configuration ---
train_file = 'train_processed_final.csv'  # Path to preprocessed training data
test_file = 'test_processed_final.csv'    # Path to preprocessed test data
sample_submission_file = 'sample_submission.csv' # To get the correct ID format/output structure
submission_dir = 'submissions'            # Directory to save predictions
# target_column = 'target'  # Removed fixed target column name
N_SPLITS = 5 # Number of folds for cross-validation

# --- Ensure submission directory exists ---
os.makedirs(submission_dir, exist_ok=True)

# --- Load Data ---
print("Loading data...")
try:
    if os.path.exists(train_file):
        df_train = pd.read_csv(train_file)
        print(f"Loaded training data: {df_train.shape}")
    else:
        print(f"Error: {train_file} not found. Please ensure preprocessed data is available.")
        exit()

    if os.path.exists(test_file):
        df_test = pd.read_csv(test_file)
        print(f"Loaded test data: {df_test.shape}")
    else:
        print(f"Error: {test_file} not found. Please ensure preprocessed test data is available.")
        exit()

    if os.path.exists(sample_submission_file):
        sample_submission = pd.read_csv(sample_submission_file)
        print(f"Loaded sample submission: {sample_submission.shape}")
    else:
        print(f"Warning: {sample_submission_file} not found. Using test set index for submission.")
        sample_submission = None

except FileNotFoundError as e:
    print(f"File not found: {e}")
    exit()

# --- Prepare Features and Target ---
# Identify the target column as the LAST column in the training set
target_column = df_train.columns[-1]  # Get the name of the last column
print(f"Identified target column: '{target_column}'")

if target_column not in df_train.columns:
    print(f"Error: Target column '{target_column}' (assumed to be the last column) not found in training data.")
    print(f"Available columns: {list(df_train.columns)}")
    exit()

X = df_train.drop(columns=[target_column])
y = df_train[target_column]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# --- Determine Feature Names from Training Set ---
feature_names = X.columns.tolist()
print(f"Feature names for training/prediction: {feature_names}")

# --- Identify the ID column for submission ---
# Use the ID from sample submission if available, otherwise use test set index
if sample_submission is not None and len(sample_submission.columns) > 0:
    id_column = sample_submission.columns[0] # Assuming the first column is the ID
    # Load the ID values from the sample submission file
    test_ids = sample_submission[id_column]
    print(f"Using ID column '{id_column}' from sample submission.")
else:
    # Use the test set's index as the ID if no sample submission is provided
    # Check if 'Id' exists in test set, otherwise use index
    if 'Id' in df_test.columns:
        test_ids = df_test['Id']
        print(f"Using 'Id' column from test set.")
    else:
        test_ids = df_test.index
        id_column = 'index' # Placeholder name for index if needed later
        print(f"Using test set index as ID column.")

# --- Prepare Test Set Features ---
# Remove the target column (if it exists) and any non-feature columns (like Id) from the test set
# Only keep the columns that match the training features
# Check if the expected features exist in the test set
missing_features = [col for col in feature_names if col not in df_test.columns]
if missing_features:
    print(f"Error: The following features from the training set are missing in the test set: {missing_features}")
    exit()

# Select only the relevant features for prediction
X_test = df_test[feature_names] # This ensures only training features are used

print(f"Test features (X_test) shape: {X_test.shape}")

# --- Determine Problem Type (Classification vs Regression) ---
# Check the data type of the target variable
if y.dtype == 'object' or isinstance(y.dtype, pd.CategoricalDtype) or len(y.unique()) <= 20: # Heuristic for classification, updated deprecation warning
    problem_type = 'classification'
    print(f"Detected target type as categorical/discrete. Assuming {problem_type}.")
    model_definitions = {
        'NaiveBayes': GaussianNB(),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGB': xgb.XGBClassifier(random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5)
        # Removed SVR for classification
    }
    scoring = ['accuracy', 'f1_macro', 'roc_auc_ovr'] # Scoring metrics for classification
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Use Stratified KFold for classification
else:
    problem_type = 'regression'
    print(f"Detected target type as numeric/continuous. Assuming {problem_type}.")
    model_definitions = {
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
        'SVR': SVR(),
        'XGB': xgb.XGBRegressor(random_state=42),
        'KNN': KNeighborsRegressor(n_neighbors=5)
        # Removed Naive Bayes for regression (less common, potentially inappropriate)
    }
    scoring = ['neg_mean_squared_error', 'r2'] # Scoring metrics for regression
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Use regular KFold for regression


# --- Train Models, Perform Cross-Validation, and Generate Predictions ---
for name, model in model_definitions.items():
    print(f"\n--- Training and Cross-Validating {name} ---")
    try:
        # Perform cross-validation
        print(f"  Performing {N_SPLITS}-Fold Cross-Validation...")
        cv_results = {}
        for metric in scoring:
            scores = cross_val_score(model, X, y, cv=cv, scoring=metric)
            cv_results[metric] = {
                'mean': scores.mean(),
                'std': scores.std()
            }
            print(f"    {metric}: Mean={cv_results[metric]['mean']:.4f}, Std={cv_results[metric]['std']:.4f}")

        # Fit the model on the entire training set
        model.fit(X, y)
        print(f"  Predicting on test set with {name}...")
        # Use the correctly formatted test features (X_test)
        predictions = model.predict(X_test)

        # Create submission dataframe
        submission_df = pd.DataFrame({
            id_column: test_ids, # Use the ID column determined earlier
            'target': predictions # Use 'target' as the column name for predictions in submission
        })

        # Save predictions to CSV in the submission directory
        submission_path = os.path.join(submission_dir, f'submission_{name.lower()}.csv')
        submission_df.to_csv(submission_path, index=False)
        print(f"  Predictions for {name} saved to {submission_path}")

    except Exception as e:
        print(f"  Error training/cross-validating/predicting with {name}: {e}")

print("\nAll models trained, cross-validated, and predictions saved (if successful). Check the 'submissions' folder.")

Loading data...
Loaded training data: (14396, 15)
Loaded test data: (3600, 15)
Identified target column: 'Class'
Features (X) shape: (14396, 14)
Target (y) shape: (14396,)
Feature names for training/prediction: ['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']
Using 'Id' column from test set.
Test features (X_test) shape: (3600, 14)
Detected target type as categorical/discrete. Assuming classification.

--- Training NaiveBayes ---
  Predicting with NaiveBayes...
  Predictions for NaiveBayes saved to submissions\submission_naivebayes.csv

--- Training RandomForest ---
  Predicting with RandomForest...
  Predictions for RandomForest saved to submissions\submission_randomforest.csv

--- Training XGB ---
  Predicting with XGB...
  Predictions for XGB saved to submissions\submission_xgb.csv

--- Training KNN ---
  Predicting with KNN...
  Predictions f